# TB Portals - Kantipudi **A3** baseline (direct Timika regressor)

One DenseNet121 regresses Timika directly (sigmoid x140, MSE). Same Table-3 country-segregated split as ALP. **Attach these Kaggle datasets before running:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (438/438), done.


## Install deps

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.7 MB/s eta 0:00:00
deps installed


## Paths

Edit dataset slugs if yours differ.

In [3]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_a3"
os.makedirs(OUT_DIR, exist_ok=True)
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [5]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main
argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))

[crops] device=cuda:Tesla T4
[crops] 0/5010 cached; generating the remaining 5010.
[crops] loaded fine-tuned lung decoder: /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt
[crops] 200/5010 (lung=200, fallback=0)
[crops] 400/5010 (lung=400, fallback=0)
[crops] 600/5010 (lung=600, fallback=0)
[crops] 800/5010 (lung=800, fallback=0)
[crops] 1000/5010 (lung=1000, fallback=0)
[crops] 1200/5010 (lung=1200, fallback=0)
[crops] 1400/5010 (lung=1400, fallback=0)
[crops] 1600/5010 (lung=1600, fallback=0)
[crops] 1800/5010 (lung=1800, fallback=0)
[crops] 2000/5010 (lung=2000, fallback=0)
[crops] 2200/5010 (lung=2200, fallback=0)
[crops] 2400/5010 (lung=2400, fallback=0)
[crops] 2600/5010 (lung=2600, fallback=0)
[crops] 2800/5010 (lung=2800, fallback=0)
[crops] 3000/5010 (lung=3000, fallback=0)
[crops] 3200/5010 (lung=3200, fallback=0)
[crops] 3400/5010 (lung=3400, fallback=0)
[crops] 3600/5010 (lung=3600, fallback=0)
[crops] 3800/5010 (lung=3800, fallback=0)
[

## 3 - Full A3 run (~1.5-2 h)

In [6]:
from src.training.train_a3_direct import main as a3_main
a3_main(['--manifest', PAPER_MANIFEST, '--crops-dir', CROPS_DIR, '--out-dir', OUT_DIR,
         '--held-outs','Romania','Moldova','Kazakhstan','--seeds','0','1','2',
         '--epochs','30','--batch-size','60','--accum-steps','5','--num-workers','2'])

[paper-a3] device=cuda
[paper-a3] input = lung-crop /kaggle/working/crops

===== A3  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[A3] train=3832 val=958 test=220
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 141MB/s]


  [A3] epoch 00 train_mse=0.073103 val_mse=0.039245
  [A3] epoch 01 train_mse=0.040920 val_mse=0.036931
  [A3] epoch 02 train_mse=0.034741 val_mse=0.032168
  [A3] epoch 03 train_mse=0.031240 val_mse=0.031468
  [A3] epoch 04 train_mse=0.029693 val_mse=0.031390
  [A3] epoch 05 train_mse=0.027968 val_mse=0.032126
  [A3] epoch 06 train_mse=0.026059 val_mse=0.032803
  [A3] epoch 07 train_mse=0.023767 val_mse=0.030118
  [A3] epoch 08 train_mse=0.026633 val_mse=0.033466
  [A3] epoch 09 train_mse=0.023983 val_mse=0.032547
  [A3] epoch 10 train_mse=0.023203 val_mse=0.030980
  [A3] epoch 11 train_mse=0.021324 val_mse=0.031534
  [A3] epoch 12 train_mse=0.019076 val_mse=0.031940
  [A3] epoch 13 train_mse=0.017842 val_mse=0.037485
  [A3] epoch 14 train_mse=0.020257 val_mse=0.033610
  [A3] epoch 15 train_mse=0.017595 val_mse=0.033221
  [A3] epoch 16 train_mse=0.016821 val_mse=0.034057
  [A3] epoch 17 train_mse=0.015530 val_mse=0.030798
  [A3] epoch 18 train_mse=0.013334 val_mse=0.032645
  [A3] epoch

## 4 - Save outputs

In [7]:
!cd /kaggle/working && zip -j results_a3.zip checkpoints/paper_a3/results_a3.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q checkpoints_a3.zip checkpoints/paper_a3
print("Saved: results_a3.zip, checkpoints_a3.zip")

  adding: results_a3.csv (deflated 49%)
  adding: tbportals_manifest_paper.csv (deflated 77%)
Saved: results_a3.zip, checkpoints_a3.zip
